In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
def generate_sequence(seq_len):
    x = np.random.randint(0, 10, size=seq_len, dtype=np.int64)
    first = x[0]

    y = np.empty_like(x)
    y[0] = x[0]
    y[1:] = (x[1:] + first) % 10

    return x, y

In [3]:
x, y = generate_sequence(10)
print('x:', x)
print('y:', y)

x: [6 3 0 7 7 2 1 1 9 7]
y: [6 9 6 3 3 8 7 7 5 3]


In [4]:
class DigitSequenceDataset(Dataset):
    def __init__(self, num_samples, seq_len):
        self.x_data = []
        self.y_data = []

        for _ in range(num_samples):
            x, y = generate_sequence(seq_len)
            self.x_data.append(x)
            self.y_data.append(y)

        self.x_data = torch.tensor(np.array(self.x_data), dtype=torch.long)
        self.y_data = torch.tensor(np.array(self.y_data), dtype=torch.long)

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

In [5]:
seq_len = 20
train_dataset = DigitSequenceDataset(num_samples=10000, seq_len=seq_len)
test_dataset = DigitSequenceDataset(num_samples=2000, seq_len=seq_len)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
class SeqModel(nn.Module):
    def __init__(self, model_type='RNN', embed_dim=16, hidden_dim=32, num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=10, embedding_dim=embed_dim)

        if model_type == 'RNN':
            self.rnn = nn.RNN(
                input_size=embed_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True
            )
        elif model_type == 'LSTM':
            self.rnn = nn.LSTM(
                input_size=embed_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True
            )
        elif model_type == 'GRU':
            self.rnn = nn.GRU(
                input_size=embed_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True
            )
        else:
            raise ValueError('model_type must be one of: RNN, LSTM, GRU')

        self.fc = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.rnn(emb)
        logits = self.fc(out)
        return logits

In [7]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

device

'mps'

In [8]:
def evaluate_model(model, data_loader, criterion=None):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)

            if criterion is not None:
                loss = criterion(
                    logits.reshape(-1, 10),
                    y_batch.reshape(-1)
                )
                total_loss += loss.item()

            preds = logits.argmax(dim=-1)
            total_correct += (preds == y_batch).sum().item()
            total_count += y_batch.numel()

    avg_loss = total_loss / len(data_loader) if criterion is not None else None
    acc = total_correct / total_count
    return avg_loss, acc

In [9]:
def train_model(model, train_loader, test_loader, epochs=10, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(x_batch)  # [B, T, 10]

            loss = criterion(
                logits.reshape(-1, 10),
                y_batch.reshape(-1)
            )

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            preds = logits.argmax(dim=-1)
            train_correct += (preds == y_batch).sum().item()
            train_total += y_batch.numel()

        train_acc = train_correct / train_total

        test_loss, test_acc = evaluate_model(model, test_loader, criterion)

        print(
            f'Epoch {epoch:02d} | '
            f'train_loss={train_loss / len(train_loader):.4f} | '
            f'train_acc={train_acc:.4f} | '
            f'test_loss={test_loss:.4f} | '
            f'test_acc={test_acc:.4f}'
        )
    return model

In [10]:
rnn_model = SeqModel(model_type='RNN', embed_dim=16, hidden_dim=32)
rnn_model = train_model(rnn_model, train_loader, test_loader, epochs=10, lr=1e-3)

Epoch 01 | train_loss=2.2988 | train_acc=0.1303 | test_loss=2.2961 | test_acc=0.1395
Epoch 02 | train_loss=2.2779 | train_acc=0.1443 | test_loss=2.2458 | test_acc=0.1419
Epoch 03 | train_loss=2.1885 | train_acc=0.1621 | test_loss=2.1380 | test_acc=0.1652
Epoch 04 | train_loss=2.0946 | train_acc=0.1806 | test_loss=2.0703 | test_acc=0.1898
Epoch 05 | train_loss=2.0419 | train_acc=0.2003 | test_loss=2.0355 | test_acc=0.2169
Epoch 06 | train_loss=2.0154 | train_acc=0.2106 | test_loss=2.0164 | test_acc=0.2100
Epoch 07 | train_loss=2.0023 | train_acc=0.2186 | test_loss=2.0048 | test_acc=0.2192
Epoch 08 | train_loss=1.9908 | train_acc=0.2256 | test_loss=1.9960 | test_acc=0.2130
Epoch 09 | train_loss=1.9820 | train_acc=0.2273 | test_loss=1.9899 | test_acc=0.2284
Epoch 10 | train_loss=1.9764 | train_acc=0.2318 | test_loss=1.9845 | test_acc=0.2288


In [11]:
lstm_model = SeqModel(model_type='LSTM', embed_dim=16, hidden_dim=32)
lstm_model = train_model(lstm_model, train_loader, test_loader, epochs=10, lr=1e-3)

Epoch 01 | train_loss=2.2995 | train_acc=0.1257 | test_loss=2.2957 | test_acc=0.1414
Epoch 02 | train_loss=2.2809 | train_acc=0.1518 | test_loss=2.2196 | test_acc=0.2036
Epoch 03 | train_loss=1.8943 | train_acc=0.2746 | test_loss=1.5829 | test_acc=0.3281
Epoch 04 | train_loss=1.4754 | train_acc=0.3514 | test_loss=1.3813 | test_acc=0.3797
Epoch 05 | train_loss=1.2801 | train_acc=0.4496 | test_loss=1.1864 | test_acc=0.4845
Epoch 06 | train_loss=1.0731 | train_acc=0.5538 | test_loss=0.9634 | test_acc=0.6359
Epoch 07 | train_loss=0.8412 | train_acc=0.6857 | test_loss=0.7460 | test_acc=0.7117
Epoch 08 | train_loss=0.6711 | train_acc=0.7338 | test_loss=0.6048 | test_acc=0.7518
Epoch 09 | train_loss=0.5384 | train_acc=0.7982 | test_loss=0.4568 | test_acc=0.8759
Epoch 10 | train_loss=0.3681 | train_acc=0.9419 | test_loss=0.2919 | test_acc=0.9790


In [12]:
gru_model = SeqModel(model_type='GRU', embed_dim=16, hidden_dim=32)
gru_model = train_model(gru_model, train_loader, test_loader, epochs=10, lr=1e-3)

Epoch 01 | train_loss=2.2985 | train_acc=0.1318 | test_loss=2.2941 | test_acc=0.1403
Epoch 02 | train_loss=2.2867 | train_acc=0.1456 | test_loss=2.2750 | test_acc=0.1458
Epoch 03 | train_loss=2.2462 | train_acc=0.1620 | test_loss=2.1923 | test_acc=0.1897
Epoch 04 | train_loss=1.9755 | train_acc=0.2784 | test_loss=1.6986 | test_acc=0.3312
Epoch 05 | train_loss=1.5336 | train_acc=0.3562 | test_loss=1.4398 | test_acc=0.3888
Epoch 06 | train_loss=1.3558 | train_acc=0.4503 | test_loss=1.3052 | test_acc=0.5056
Epoch 07 | train_loss=1.1599 | train_acc=0.5939 | test_loss=1.0457 | test_acc=0.6684
Epoch 08 | train_loss=0.8904 | train_acc=0.7611 | test_loss=0.7459 | test_acc=0.8417
Epoch 09 | train_loss=0.6153 | train_acc=0.8866 | test_loss=0.5054 | test_acc=0.9127
Epoch 10 | train_loss=0.4174 | train_acc=0.9346 | test_loss=0.3448 | test_acc=0.9512


In [13]:
def predict_sequence(model, x_seq):
    model.eval()

    x_tensor = torch.from_numpy(x_seq).long().unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x_tensor)
        preds = logits.argmax(dim=-1).squeeze(0).cpu().numpy()

    return preds

In [14]:
x, y_true = generate_sequence(seq_len=10)
y_pred = predict_sequence(gru_model, x)

print('x      :', x)
print('y_true :', y_true)
print('y_pred :', y_pred)

x      : [5 0 1 4 3 0 2 8 0 6]
y_true : [5 5 6 9 8 5 7 3 5 1]
y_pred : [5 5 6 9 8 5 7 3 5 1]
